# Exercício 06 — PySpark: UDFs (User Defined Functions)

**Tópico:** PySpark — UDF, Pandas UDF, transformações customizadas

---

## Setup
Faça o upload de `funcionarios.csv` e `vendas.csv` para o DBFS.

---

## Exercício 1 — UDF simples: classificar salário
Leia `funcionarios.csv`.  
Crie uma UDF chamada `classificar_salario` que recebe o salário e retorna:
- `'Junior'` se salário < 7000
- `'Pleno'` se salário entre 7000 e 11000
- `'Senior'` se salário > 11000

Aplique a UDF criando uma nova coluna `nivel` no DataFrame.

---

## Exercício 2 — UDF: formatar nome
Crie uma UDF chamada `iniciais` que recebe o nome completo (ex: `"Ana Souza"`) e retorna as iniciais (ex: `"A.S."`).  
Aplique no DataFrame de funcionários criando a coluna `iniciais`.

---

## Exercício 3 — UDF: calcular anos na empresa
Crie uma UDF que recebe `data_contratacao` (string no formato `YYYY-MM-DD`) e retorna quantos **anos completos** o funcionário tem de empresa até hoje.  
Crie a coluna `anos_empresa` no DataFrame.

---

## Exercício 4 — Pandas UDF: desconto sobre valor
Leia `vendas.csv`.  
Crie uma **Pandas UDF** chamada `aplicar_desconto` que recebe a coluna `valor` (DoubleType) e aplica um desconto de 10%, retornando o novo valor.  
Adicione a coluna `valor_com_desconto` ao DataFrame.

---

## Exercício 5 — Comparar UDF vs função nativa
Para a operação do exercício 4 (aplicar 10% de desconto),  
implemente a mesma lógica usando **apenas funções nativas do PySpark** (sem UDF).  
Compare os dois resultados e reflita: quando vale usar UDF e quando não vale?


In [0]:
FUNCIONARIOS_PATH = "/Workspace/Users/leocodedev@outlook.com/pyspark/notebooks/exercise/data/funcionarios.csv"
VENDAS_PATH = "/Workspace/Users/leocodedev@outlook.com/pyspark/notebooks/exercise/data/vendas.csv"

In [0]:
from pyspark.sql.functions import udf, col, year

In [0]:
# Exercício 1
@udf
def classificar_salario(salario):
    classificacao = ""
    if salario < 7000: classificacao = "Junior"
    elif salario > 7000 and salario < 11000: classificacao = "Pleno"
    else: classificacao = "Senior"
    
    return classificacao

df1 = spark.read.csv(FUNCIONARIOS_PATH, header=True, inferSchema=True)
df1 = df1.withColumn("nivel", classificar_salario(col("salario")))

display(df1)


In [0]:
# Exercício 2
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

df2 = spark.read.csv(FUNCIONARIOS_PATH, header=True, inferSchema=True)

@udf(StringType())
def iniciais(nome: str):
    return ".".join(nome[:3])

df2 = df2.withColumn("iniciais", iniciais(col("nome")))
display(df2)


In [0]:
# Exercício 3
from datetime import datetime

df3 = spark.read.csv(FUNCIONARIOS_PATH, header=True, inferSchema=True)

@udf
def data_contratacao(data):
    data_atual = datetime.now().year
    anos_de_trabalho = data_atual - data
    return anos_de_trabalho

df3 = df3.withColumn("anos_empresa", data_contratacao(year(col("data_contratacao"))))
display(df3)

In [0]:
# Exercício 4
#Leia vendas.csv.
#Crie uma Pandas UDF chamada aplicar_desconto que recebe a coluna valor (DoubleType) e aplica um desconto de 10%, #retornando o novo valor.
#Adicione a coluna valor_com_desconto ao DataFrame.
from pyspark.sql.types import DoubleType
df4 = spark.read.csv(VENDAS_PATH, header=True, inferSchema=True)

@udf(DoubleType())
def aplicar_desconto(valor):
    desconto = 10 / 100
    return valor - (valor * desconto)
df4 = df4.withColumn("valor_com_desconto", aplicar_desconto(col("valor")))
display(df4)



In [0]:
# Exercício 5
#Para a operação do exercício 4 (aplicar 10% de desconto),  
#implemente a mesma lógica usando **apenas funções nativas do PySpark** (sem UDF).  
#Compare os dois resultados e reflita: quando vale usar UDF e quando não vale?

df5 = spark.read.csv(VENDAS_PATH, header=True, inferSchema=True)

df5 = df5.withColumn("valor_com_desconto", col("valor") - (col("valor") * 10/100))
display(df5)
